# Stage 07-A: Stage 0 Sanity Check — PhoBERT + ResNet-50 Vietnamese Alignment

**Purpose:** Verify that PhoBERT CLS embeddings and ResNet-50 image features produce
meaningful pairwise cosine similarities on Vietnamese image-text pairs **before any training**.
If positive pairs (same article) don't score consistently higher than in-batch negatives even
after random projection, the contrastive pretraining in 07b faces a weak signal problem.

This is the cheapest falsification of the entire Path B premise (§7 of the research plan).

## What this notebook does

1. Load a small sample of image-text pairs from `stage06d_cache`
2. Run PhoBERT forward pass to get CLS embeddings (no training)
3. Load COOLANT image features (64-dim) as the visual signal
4. Apply **random** FC projection to shared `proj_dim=128` for both modalities
5. Compute pairwise cosine similarity matrix — visualise as heatmap
6. Check Unicode NFC normalisation consistency on Vietnamese text
7. Report `has_valid_image` statistics (pairs where pair_count > 0)
8. **Decision rule**: if median positive-pair sim significantly exceeds median negative-pair sim
   (even with random projection), it confirms raw feature quality is sufficient for Stage 1.

## Prerequisites

- `06d_prepare_dataset.ipynb` — builds `stage06d_cache/{train,dev,test}.h5`
- `vinai/phobert-base-v2` available (HuggingFace Hub or local cache)

## Outputs

- `training/stage07a_results/cosine_sim_heatmap.png`
- `training/stage07a_results/has_valid_image_stats.json`
- `training/stage07a_results/nfc_audit.csv`
- Console: decision recommendation for 07b

In [ ]:
# ─── Environment Setup (do not edit) ─────────────────────────────────────────
import os, sys
from pathlib import Path


def _detect_platform():
    try:
        import google.colab  # noqa: F401
        return 'colab', False
    except ImportError:
        pass
    if Path('/workspace').exists() and os.environ.get('VAST_CONTAINERLABEL'):
        return 'vastai', False
    if Path('/workspace').exists():
        return 'vastai', True
    if sys.platform == 'win32': return 'windows', False
    if sys.platform == 'darwin': return 'mac', False
    return None, True


PLATFORM, _uncertain = _detect_platform()
if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

try:
    _nb_path = Path(__file__).resolve()
except NameError:
    _nb_path = Path.cwd()

PROJECT_ROOT = (
    Path('/content/drive/MyDrive/Thesis_Final/fake-news-detection') if PLATFORM == 'colab'
    else _nb_path.parents[1]
)
sys.path.insert(0, str(PROJECT_ROOT))

_env_map = {
    'colab':   PROJECT_ROOT / '.env.colab',
    'vastai':  PROJECT_ROOT / '.env.vastai',
    'windows': PROJECT_ROOT / '.env.windows',
    'mac':     PROJECT_ROOT / '.env.mac',
}
_env_file = _env_map.get(PLATFORM, PROJECT_ROOT / '.env')
if not _env_file.exists():
    _env_file = PROJECT_ROOT / '.env'

from dotenv import load_dotenv
load_dotenv(_env_file, override=True)
from src.utils.env_utils import get_data_root
DATA_ROOT = get_data_root()
print(f'Platform : {PLATFORM}  DATA_ROOT: {DATA_ROOT}')

In [ ]:
# ── Step 1: Configuration ─────────────────────────────────────────────────────
CONFIG = {
    'paths': {
        'stage06d_cache_dir': DATA_ROOT / 'training' / 'stage06d_cache',
        'results_dir':        DATA_ROOT / 'training' / 'stage07a_results',
    },
    'phobert': {
        'model_name':  'vinai/phobert-base-v2',
        'max_seq_len': 128,
    },
    'sanity': {
        'n_samples':  30,      # pairs to inspect (20-30 per research plan §4 Stage 0)
        'split':      'dev',   # which split to sample from
        'proj_dim':   128,     # shared projection dim (matches COOLANT)
        'seed':       42,
        'image_dim':  64,      # COOLANT clip_embed_dim from stage06d_cache
        'text_dim':   768,     # PhoBERT-base hidden size
    },
}
CONFIG['paths']['results_dir'].mkdir(parents=True, exist_ok=True)
print(f'Results dir: {CONFIG["paths"]["results_dir"]}')

In [ ]:
# ── Step 2: Dependencies ──────────────────────────────────────────────────────
import importlib, unicodedata, random, json, gc
from datetime import datetime

_required = [
    ('torch', 'torch'), ('h5py', 'h5py'), ('numpy', 'numpy'),
    ('pandas', 'pandas'), ('matplotlib', 'matplotlib'),
    ('seaborn', 'seaborn'), ('transformers', 'transformers'),
]
_missing = [pkg for mod, pkg in _required if importlib.util.find_spec(mod) is None]
if _missing:
    raise RuntimeError(f'Missing packages: {_missing}. Install them and restart.')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

random.seed(CONFIG['sanity']['seed'])
np.random.seed(CONFIG['sanity']['seed'])
torch.manual_seed(CONFIG['sanity']['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}  PyTorch: {torch.__version__}')

In [ ]:
# ── Step 3: Load Sample Pairs from stage06d_cache ─────────────────────────────
split     = CONFIG['sanity']['split']
n_samples = CONFIG['sanity']['n_samples']
cache_p   = CONFIG['paths']['stage06d_cache_dir'] / f'{split}.h5'

if not cache_p.exists():
    raise FileNotFoundError(
        f'Cache not found: {cache_p}\nRun 06d_prepare_dataset.ipynb first.'
    )

with h5py.File(cache_p, 'r') as f:
    n_total   = int(f.attrs['n_articles'])
    max_pairs = int(f.attrs['max_pairs'])
    imgs_all  = f['image_features'][:]   # [N, max_pairs, 64]
    cnts_all  = f['pair_counts'][:]       # [N]
    lbls_all  = f['labels'][:]            # [N]
    stmts_all = [str(s) for s in f['statements'][:]]
    ctxs_all  = [str(s) for s in f['contexts'][:]]

# Sample n_samples instances, stratified by label
indices = list(range(n_total))
random.shuffle(indices)
sample_idx = indices[:n_samples]

sample = [{
    'idx':            i,
    'statement':      stmts_all[i],
    'context':        ctxs_all[i],
    'image':          imgs_all[i].astype(np.float32),   # [max_pairs, 64]
    'pair_count':     int(cnts_all[i]),
    'label':          int(lbls_all[i]),
    'has_valid_image': int(cnts_all[i]) > 0,
} for i in sample_idx]

print(f'Loaded {len(sample)} samples from split={split} (total={n_total}, max_pairs={max_pairs})')
label_names = {0: 'Real', 1: 'Fake', 2: 'NEI'}
for lbl, name in label_names.items():
    cnt = sum(1 for s in sample if s['label'] == lbl)
    vi  = sum(1 for s in sample if s['label'] == lbl and s['has_valid_image'])
    print(f'  {name}: {cnt} total, {vi} with image')

In [ ]:
# ── Step 4: Vietnamese Unicode NFC Normalisation Audit ────────────────────────
# Vietnamese diacritics may be stored as NFC or NFD (decomposed), silently producing
# different token IDs for visually identical text. Check before any tokenisation.

def nfc_diff(text):
    nfc = unicodedata.normalize('NFC', text)
    return text != nfc, len(text), len(nfc)

audit_rows = []
for s in sample:
    for field in ('statement', 'context'):
        txt = s[field]
        differs, orig_len, nfc_len = nfc_diff(txt)
        if differs:
            audit_rows.append({
                'idx':     s['idx'],
                'field':   field,
                'orig_len': orig_len,
                'nfc_len':  nfc_len,
                'snippet':  txt[:80],
            })

if audit_rows:
    df_nfc = pd.DataFrame(audit_rows)
    nfc_path = CONFIG['paths']['results_dir'] / 'nfc_audit.csv'
    df_nfc.to_csv(nfc_path, index=False)
    print(f'[WARN] {len(audit_rows)} fields differ from NFC form — saved to {nfc_path}')
    print('  Apply unicodedata.normalize("NFC", text) in preprocessing before tokenisation.')
else:
    print('[OK] All sampled text is already NFC-normalised.')

# Normalise in-place for remainder of this notebook
for s in sample:
    s['statement'] = unicodedata.normalize('NFC', s['statement'])
    s['context']   = unicodedata.normalize('NFC', s['context'])

In [ ]:
# ── Step 5: PhoBERT CLS Embeddings (no training) ─────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG['phobert']['model_name'], use_fast=True
)
phobert = AutoModel.from_pretrained(CONFIG['phobert']['model_name']).to(device)
phobert.eval()

texts = [s['statement'] for s in sample]
enc = tokenizer(
    texts,
    max_length=CONFIG['phobert']['max_seq_len'],
    padding=True,
    truncation=True,
    return_tensors='pt',
).to(device)

with torch.no_grad():
    out = phobert(**enc)
    text_cls = out.last_hidden_state[:, 0, :]  # [N, 768]

text_cls = F.normalize(text_cls, dim=-1)  # L2-normalise
print(f'PhoBERT CLS shape: {text_cls.shape}  norm={text_cls.norm(dim=-1).mean():.4f}')

# Free GPU memory — we don't need the full model for the rest of this notebook
del phobert, enc, out
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

In [ ]:
# ── Step 6: Image Features + Random Projection ───────────────────────────────
# Take the MEAN over valid image pairs (pair_count > 0); fall back to zeros.
# Then project both text and image to shared proj_dim with RANDOM FC weights
# (untrained). If even random projection shows a signal, features have structure.

proj_dim  = CONFIG['sanity']['proj_dim']
image_dim = CONFIG['sanity']['image_dim']
text_dim  = CONFIG['sanity']['text_dim']

# Mean-pool valid image pairs
img_vecs = []
for s in sample:
    pc = s['pair_count']
    if pc > 0:
        vec = s['image'][:pc].mean(axis=0)  # [64]
    else:
        vec = np.zeros(image_dim, dtype=np.float32)
    img_vecs.append(vec)

img_tensor = torch.tensor(np.stack(img_vecs), dtype=torch.float32).to(device)  # [N, 64]
img_tensor = F.normalize(img_tensor, dim=-1)

torch.manual_seed(CONFIG['sanity']['seed'])
text_proj  = nn.Linear(text_dim,  proj_dim, bias=False).to(device)
image_proj = nn.Linear(image_dim, proj_dim, bias=False).to(device)
nn.init.xavier_uniform_(text_proj.weight)
nn.init.xavier_uniform_(image_proj.weight)

with torch.no_grad():
    e_t = F.normalize(text_proj(text_cls), dim=-1)    # [N, proj_dim]
    e_v = F.normalize(image_proj(img_tensor), dim=-1)  # [N, proj_dim]

print(f'e_t: {e_t.shape}  e_v: {e_v.shape}')
print(f'has_valid_image: {sum(s["has_valid_image"] for s in sample)}/{len(sample)}')

In [ ]:
# ── Step 7: Pairwise Cosine Similarity Matrix ─────────────────────────────────
# Image-to-text similarity: sim[i,j] = cos(e_v[i], e_t[j])
# Diagonal = positive pairs (same article), off-diagonal = in-batch negatives

with torch.no_grad():
    sim_matrix = (e_v @ e_t.T).cpu().numpy()  # [N, N]

# Separate positive (diagonal) and negative (off-diagonal) similarities
pos_sims = sim_matrix.diagonal()
neg_mask  = ~np.eye(len(sample), dtype=bool)

# Only count pairs where image is valid (pair_count > 0)
valid_mask = np.array([s['has_valid_image'] for s in sample])
valid_pos  = pos_sims[valid_mask]
valid_neg  = sim_matrix[np.ix_(valid_mask, ~valid_mask)].flatten()
neg_all    = sim_matrix[neg_mask]

print(f'Positive pairs (has_valid_image=True):')
print(f'  mean={valid_pos.mean():.4f}  median={np.median(valid_pos):.4f}  std={valid_pos.std():.4f}')
print(f'Negative pairs:')
print(f'  mean={neg_all.mean():.4f}  median={np.median(neg_all):.4f}  std={neg_all.std():.4f}')
print(f'Delta (pos - neg mean): {valid_pos.mean() - neg_all.mean():+.4f}')

In [ ]:
# ── Step 8: Heatmap Visualisation ────────────────────────────────────────────
label_ticks = [f'{label_names[s["label"]][0]}{"*" if s["has_valid_image"] else ""}'
               for s in sample]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: full cosine sim matrix
sns.heatmap(
    sim_matrix, ax=axes[0], cmap='RdYlGn', center=0,
    xticklabels=label_ticks, yticklabels=label_ticks,
    linewidths=0.3, linecolor='gray', vmin=-1, vmax=1,
)
axes[0].set_title('Image-Text Cosine Similarity\n(* = has valid image, diagonal = positive pairs)',
                  fontsize=11)
axes[0].set_xlabel('Text index'); axes[0].set_ylabel('Image index')

# Right: distribution of positive vs negative similarities
axes[1].hist(neg_all, bins=30, alpha=0.6, color='#EF4444', label='Negative pairs (all)', density=True)
if len(valid_pos) > 0:
    axes[1].hist(valid_pos, bins=10, alpha=0.8, color='#22C55E', label='Positive pairs (image valid)', density=True)
axes[1].axvline(valid_pos.mean(),  color='#15803D', linestyle='--', linewidth=1.5, label=f'pos mean={valid_pos.mean():.3f}')
axes[1].axvline(neg_all.mean(),   color='#DC2626', linestyle='--', linewidth=1.5, label=f'neg mean={neg_all.mean():.3f}')
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_ylabel('Density')
axes[1].set_title('Positive vs Negative Pair Similarity\n(random projection, no training)')
axes[1].legend(fontsize=9)

plt.suptitle(
    f'Stage 07-A Sanity Check — PhoBERT + ResNet-64  (n={len(sample)}, split={split})',
    fontsize=13
)
plt.tight_layout()
p = CONFIG['paths']['results_dir'] / 'cosine_sim_heatmap.png'
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f'Saved: {p}')

In [ ]:
# ── Step 9: has_valid_image Statistics (full split) ───────────────────────────
# Report across the full split so we know the usable multimodal subset size
# for Stage 1 contrastive pretraining.

with h5py.File(cache_p, 'r') as f:
    all_cnts = f['pair_counts'][:]
    all_lbls = f['labels'][:]

total = len(all_cnts)
with_image = int((all_cnts > 0).sum())
label_hist = {}
for lbl, name in label_names.items():
    mask_lbl = all_lbls == lbl
    label_hist[name] = {
        'total':      int(mask_lbl.sum()),
        'with_image': int((mask_lbl & (all_cnts > 0)).sum()),
    }

stats = {
    'split':            split,
    'total':            total,
    'with_valid_image': with_image,
    'pct_with_image':   round(100 * with_image / total, 1),
    'mean_pair_count':  float(all_cnts[all_cnts > 0].mean()) if with_image else 0,
    'by_label':         label_hist,
    'generated':        datetime.now().isoformat(),
}

sp = CONFIG['paths']['results_dir'] / 'has_valid_image_stats.json'
with open(sp, 'w') as f: json.dump(stats, f, indent=2)

print(f'Split: {split}  Total={total}  With image={with_image} ({stats["pct_with_image"]}%)')
for name, counts in label_hist.items():
    pct = round(100 * counts['with_image'] / counts['total'], 1) if counts['total'] else 0
    print(f'  {name}: {counts["total"]} total, {counts["with_image"]} with image ({pct}%)')
print(f'Stats saved: {sp}')

In [ ]:
# ── Step 10: Sanity Check Decision Summary ────────────────────────────────────
delta = valid_pos.mean() - neg_all.mean()

print('=' * 65)
print('STAGE 0 SANITY CHECK — DECISION SUMMARY')
print('=' * 65)
print(f'Positive pair mean cosine sim : {valid_pos.mean():.4f}')
print(f'Negative pair mean cosine sim : {neg_all.mean():.4f}')
print(f'Delta (pos - neg)             : {delta:+.4f}')
print(f'Pairs with valid image ({split}): {with_image}/{total} ({stats["pct_with_image"]}%)')
print()

SIGNAL_THRESHOLD = 0.03  # conservative: even a 3pp delta with random weights is notable

if with_image < 50:
    print('[CAUTION] Very few image pairs available for Stage 1 contrastive pretraining.')
    print('  Consider augmenting with unlabeled Vietnamese news image-text pairs')
    print('  before running 07b — Stage 1 needs sufficient in-batch negatives.')

if delta > SIGNAL_THRESHOLD:
    print('[PASS] Positive pairs score higher than negatives even with random projection.')
    print('  Raw PhoBERT + ResNet-64 features carry image-text correspondence signal.')
    print('  RECOMMENDATION: Proceed to 07b (Stage 1 contrastive pretraining).')
elif delta > 0:
    print('[MARGINAL] Positive pairs score slightly higher but delta is below threshold.')
    print(f'  ({delta:.4f} < {SIGNAL_THRESHOLD}). May be noise from small n={len(valid_pos)} valid pairs.')
    print('  RECOMMENDATION: Proceed to 07b cautiously. Monitor ITC val acc closely.')
else:
    print('[WARN] Positive pairs do NOT score higher than negatives after random projection.')
    print('  This suggests weak image-text correspondence in this domain.')
    print('  RECOMMENDATION: Investigate image sourcing before committing to Stage 1.')
    print('  Possible fallback: Path A (frozen COOLANT features + Vietnamese text model).')

print()
if audit_rows:
    print(f'[WARN] NFC normalisation required: {len(audit_rows)} field(s) had non-NFC text.')
    print('  Fix in 06d_prepare_dataset.ipynb or at Dataset load time in 07b/07c.')
else:
    print('[OK] NFC normalisation: all text already NFC-compliant.')
print('=' * 65)
print(f'Report generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')